### Verificando em qual ambiente meu Python está rodando

In [1]:
import sys
import os
print("Python Executável:", sys.executable)
print("Conda Environment:", os.environ.get('CONDA_DEFAULT_ENV'))

Python Executável: C:\Users\Bill_\anaconda3\envs\Scraping\python.exe
Conda Environment: Scraping


### Instalando as bibliotecas para WebScraping

In [8]:
#pip install requests beautifulsoup4 selenium webdriver-manager

### Bibliotecas

In [2]:
# Bibliotecas para WebScraping
import requests
'''from bs4 import BeautifulSoup''' #remover?
import selenium
import webdriver_manager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By  # Para localizar elementos
from selenium.webdriver.common.keys import Keys  # Para interagir com teclas (Enter, Tab, etc.)
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
# --------------------------------- // ---------------------------------
# Demais Bibliotecas
import pandas as pd
import re
import time

### Atenção a URL para raspagem tem filtros, pois o site apresenta apenas 100 páginas e não mostra todos itens, sendo necessário filtros

In [3]:
# Filtro de preço min 150k, apto, casa, casa de cond, sobrado, cobertura e ordem crescente
# Fiz assim para tirar os terrenos, kitnet, flat, comercial, etc...
#url = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,,,,,city,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
url = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/floradas-de-sao-jose/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Floradas%20de%20S%C3%A3o%20Jos%C3%A9,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EFloradas%20de%20Sao%20Jose,-23.21897,-45.889117,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"

# Cabeçalhos para simular um navegador real (ajuda a evitar bloqueios)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

In [4]:
# Caminho do Brave
brave = r'C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe'

# Configurações do navegador
options = Options()
options.binary_location = brave  # Define o Brave como navegador
options.add_argument("--start-maximized")  # Abre o navegador em tela cheia
options.add_argument("--disable-blink-features=AutomationControlled")  # Disfarçar o Selenium
#options.add_argument("--headless")  # Rodar sem abrir janela (se quiser ver a janela, comente essa linha)
#deixei comentado pra página não identificar que é um "robo"

In [5]:
'''# Inicializa o navegador com o ChromeDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Abre a página
driver.get(url)

# Espera alguns segundos para garantir que o conteúdo carregue
time.sleep(5)

# Verifica se a página carregou
print("Título da página:", driver.title)

# Fecha o navegador
driver.quit()'''

Título da página: Apartamentos à venda em São José dos Campos, SP - Viva Real


### Passo importante que não está no código
<br>

#### Para saber quais caracteristicas do HTML eu precisava buscar, eu...
<br>

##### * Acessei o site: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/
##### * Fui com o mouse em cima de um "bloco de anuncio de imóvel", cliquei com direito e opção *inspecionar*
##### * Passei o mouse em cima das divs para ver qual delas tinham as informações que eu queria e então anotei para uso as divs e as informações chave como descrito abaixo.
<br>

##### data-cy="rp-cardProperty-price-txt" (preço)
##### data-cy="rp-cardProperty-bedroomQuantity-txt" (quartos)
##### data-cy="rp-cardProperty-bathroomQuantity-txt" (banheiros)
##### data-cy="rp-cardProperty-propertyArea-txt" (área)



### Sucesso parcial
<br>

##### Foi encontrado apenas parte dos preços. Problemas a resolver:
##### * A página carrega os itens progressivamente a medida que é feito scroll down.
##### * Os itens não estão tudo na primeira página, além do scroll ainda tem páginação.

In [6]:
'''# Inicializa o navegador com o ChromeDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
# Abre a página
driver.get(url)

# Espera garantir o carregamento
time.sleep(5)

# Busca por todos os elementos com data-cy do preço
precos = driver.find_elements(By.CSS_SELECTOR, "[data-cy='rp-cardProperty-price-txt']")

print(f"Total de preços encontrados: {len(precos)}\n")

# Exibe os primeiros
for preco in precos[:5]:
    print(preco.text)
    print("\n")
    
# Fecha o navegador
driver.quit()'''

Total de preços encontrados: 15

R$ 205.000
Cond. R$ 190


R$ 150.000
Cond. R$ 290 • IPTU R$ 40


R$ 150.000
Preço abaixo do mercado


R$ 150.000
Cond. R$ 289


R$ 150.000
Cond. R$ 390 • IPTU R$ 100




### Atualização dinâmica superada!
<br>

#### Para carregar até o fim da página, foi feito uma função que vai até o fim da página e volta X pixels que é a variável altura_offset, então é esperado X tempo (tempo_espera) e comparado se a altura alterou, se a altura não alterar depois de max_tentativas_sem_mudanca comparações, então finaliza; Porém na prática já percebi que depois que chega nessa altura da página, a página começa a atualizar sem parar, também conferi a quantidade de dados pelo ParseHub e bateu com o total de elementos.

### Função para pecorrer a página e atualizar todos dados da página

In [5]:
def scroll_de_baixo_para_cima_limitado(driver, incremento=1000, tempo_entre_scroll=0.5, tempo_estabilizacao=3, max_scrolls_sem_carga=8):
    """
    Faz o scroll da página de baixo para cima, subindo de 'incremento' em 'incremento',
    e aguardando carregamentos dinâmicos se detectado mudança na altura da página.
    
    driver: WebDriver ativo
    incremento: Quantidade de pixels para subir a cada passo
    tempo_entre_scroll: Tempo (em segundos) entre cada scroll
    tempo_estabilizacao: Tempo extra para aguardar carregamentos se a página crescer
    max_scrolls_sem_carga: Quantidade de scrolls permitidos sem que novos dados sejam carregados
    """
    scroll_height_anterior = driver.execute_script("return document.body.scrollHeight")
    posicao_atual = scroll_height_anterior
    scrolls_sem_carregamento = 0

    while scrolls_sem_carregamento < max_scrolls_sem_carga:
        #MAX e 0 funciona se por acaso aparecer uma página que seja menor que o incremento (muito raro)
        nova_posicao = max(0, posicao_atual - incremento) 
        # Leva o scroll para o final da página menos o incremento, o zero é a horizontal
        driver.execute_script(f"window.scrollTo(0, {nova_posicao});")
        # Da uma esperada pra dar tempo de carregar
        time.sleep(tempo_entre_scroll)
        # Atualiza a posição da página, importante pra quando entrar no ELSE
        posicao_atual = nova_posicao

        # Salva o máximo da página para comparar se mudou
        scroll_height_novo = driver.execute_script("return document.body.scrollHeight")

        if scroll_height_novo > scroll_height_anterior:
            # Novos elementos carregados, reseta contagem e atualiza altura
            scroll_height_anterior = scroll_height_novo
            posicao_atual = scroll_height_novo
            scrolls_sem_carregamento = 0
            time.sleep(tempo_estabilizacao)  # Espera os novos dados carregarem
        else:
            # Nenhum carregamento, conta como um scroll "vazio" sem atualização da página
            scrolls_sem_carregamento += 1

### O que está acontecendo aqui?
<br>

#### Obs: O passo 2 serve para evitar pegar preços fora do que foi filtrado, como indicações e outros itens fora da lista principal do site.
<br>

#### 1 - Abre o navegador, acessa o URL e percorre com a função
##### uso do webdriver, get(url) e função scroll_de_baixo_para_cima_limitado
<br>

#### 2 - Salva os cards na variável (os cards são os imóveis listados)
##### na var cards
<br>

#### 3 - Retira o valor dos imóveis dos cards filtrados
##### Uso do For para percorrer e salvar na lista precos

### Testando se está capturando os valores da página

In [7]:
'''# Inicializa o navegador com o ChromeDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Acessa o site
driver.get(url)

# Aguarda o carregamento inicial
time.sleep(3)

# Executa o scroll até o final da página (robusto)
scroll_de_baixo_para_cima_limitado(driver)

# Após o scroll completo, extrai os cards principais
cards = driver.find_elements(By.CSS_SELECTOR, '[data-cy="rp-property-cd"]')

print(f"Total de cards de imóveis encontrados: {len(cards)}\n")

# Lista para armazenar os preços extraídos
precos = []

# Para cada card de imóvel, tentar extrair o preço
for card in cards:
    try:
        preco = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"]').text.strip()
        if preco:
            precos.append(preco)
    except:
        pass  # Se não encontrar preço dentro do card, ignora

print(f"Total de preços encontrados (dentro dos cards): {len(precos)}\n")

# Corrige os prints (sem usar .text em strings)
if precos:
    print("Primeiro preço encontrado:")
    print(precos[0])

    print("\nQuadrinúltimo preço encontrado:")
    print(precos[-4])
    
    print("\nTrinúltimo preço encontrado:")
    print(precos[-3])
    
    print("\nPenúltimo preço encontrado:")
    print(precos[-2])
    
    print("\nÚltimo preço encontrado:")
    print(precos[-1])
else:
    print("Nenhum preço encontrado.")

driver.quit()'''

Total de cards de imóveis encontrados: 30

Total de preços encontrados (dentro dos cards): 30

Primeiro preço encontrado:
R$ 205.000
Cond. R$ 190

Quadrinúltimo preço encontrado:
R$ 159.000

Trinúltimo preço encontrado:
R$ 159.000
Cond. R$ 180

Penúltimo preço encontrado:
R$ 159.900
Cond. R$ 190

Último preço encontrado:
R$ 160.000


### Paginação!

#### Pra começar dei inspecionar na div que contem a páginação, que contém os botões de antes e depois e os 3 números de páginas mostrados. O que vai interessar é o data-testid="next-page", que é o pra ir para a próxima, se ele não existir mais ou falhar é porque acabou as páginas, então o plano é repetir o que foi feito antes, porém em todas páginas até acabar.

### Função para fazer paginação

In [6]:
'''def coletar_paginas_completas(driver, tempo_espera_pagina=3, limite_paginas=100):
    todos_precos = []
    pagina_atual = 1

    while pagina_atual <= limite_paginas:
        print(f"\nPágina {pagina_atual}")

        scroll_de_baixo_para_cima_limitado(driver)

        # Captura os cards reais de imóveis
        cards = driver.find_elements(By.CSS_SELECTOR, '[data-cy="rp-property-cd"]')
        print(f"Encontrados {len(cards)} cards nesta página.")

        if len(cards) == 0:
            print("Nenhum card encontrado. Parando: página sem conteúdo.")
            break
        else:
            # Extrai o preço de cada card
            for card in cards:
                try:
                    preco = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"]').text.strip()
                    if preco:
                        todos_precos.append(preco)
                except NoSuchElementException:
                    continue  # Se não encontrar o preço no card, ignora

        # Tenta avançar para a próxima página
        try:
            botao_proxima = driver.find_element(By.CSS_SELECTOR, 'button[data-testid="next-page"]')
            botao_proxima.click()
            pagina_atual += 1
            time.sleep(tempo_espera_pagina)
        except Exception as e:
            print("Erro ao tentar avançar para a próxima página:", e)
            break

    return todos_precos'''

### Testar pegar preços em todas páginas

In [10]:
'''# Inicializa o navegador com o ChromeDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
# Acessa o site
driver.get(url)
time.sleep(5) # tempo de espera

precos = coletar_paginas_completas(driver)
print(f"\nTotal de preços coletados: {len(precos)}")

driver.quit()'''


Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 30 cards nesta página.

Página 4
Encontrados 30 cards nesta página.

Página 5
Encontrados 30 cards nesta página.

Página 6
Encontrados 30 cards nesta página.

Página 7
Encontrados 30 cards nesta página.

Página 8
Encontrados 30 cards nesta página.

Página 9
Encontrados 30 cards nesta página.

Página 10
Encontrados 30 cards nesta página.

Página 11
Encontrados 30 cards nesta página.

Página 12
Encontrados 30 cards nesta página.

Página 13
Encontrados 30 cards nesta página.

Página 14
Encontrados 30 cards nesta página.

Página 15
Encontrados 30 cards nesta página.

Página 16
Encontrados 30 cards nesta página.

Página 17
Encontrados 30 cards nesta página.

Página 18
Encontrados 30 cards nesta página.

Página 19
Encontrados 30 cards nesta página.

Página 20
Encontrados 30 cards nesta página.

Página 21
Encontrados 30 cards nesta página.

Página 22
Encontrados 30 cards nesta págin

### Atualizado função para paginação mas com todos dados

In [6]:
def coletar_paginas_completas(driver, tempo_espera_pagina=3, limite_paginas=100):
    dados_imoveis = []
    pagina_atual = 1

    while pagina_atual <= limite_paginas:
        print(f"\nPágina {pagina_atual}")
        scroll_de_baixo_para_cima_limitado(driver)

        cards = driver.find_elements(By.CSS_SELECTOR, '[data-cy="rp-property-cd"]')
        print(f"Encontrados {len(cards)} cards nesta página.")

        if len(cards) == 0:
            print("Nenhum card encontrado. Parando: página sem conteúdo.")
            break
        else:
            for card in cards:
                # --- DADOS DIRETOS ---
                try:
                    preco = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"] > p').text.strip()
                except NoSuchElementException:
                    preco = ''

                try:
                    rua = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-street-txt"]').text.strip()
                except NoSuchElementException:
                    rua = ''

                try:
                    area = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-propertyArea-txt"]').text.strip()
                except NoSuchElementException:
                    area = ''

                try:
                    quartos = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bedroomQuantity-txt"]').text.strip()
                except NoSuchElementException:
                    quartos = ''

                try:
                    banheiros = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bathroomQuantity-txt"]').text.strip()
                except NoSuchElementException:
                    banheiros = ''

                try:
                    vagas = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-parkingSpacesQuantity-txt"]').text.strip()
                except NoSuchElementException:
                    vagas = ''

                try:
                    link = card.find_element(By.TAG_NAME, 'a').get_attribute('href')
                except NoSuchElementException:
                    link = ''

                # --- LOCALIZAÇÃO (BAIRRO + CIDADE) ---
                try:
                    localizacao = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-location-txt"]').text.strip()
                    # O strip tira qualquer espaço que tenha ficado antes ou depois
                    bairro, cidade = [parte.strip() for parte in localizacao.split(',')]
                except NoSuchElementException:
                    bairro = ''
                    cidade = ''

                # --- CONDOMÍNIO E IPTU (TRATAMENTO DE TEXTO) ---
                try:
                    texto_completo = card.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"]').text
                    condominio = re.search(r'Cond\.\s*R\$[\s\d\.,]+', texto_completo)
                    iptu = re.search(r'IPTU\s*R\$[\s\d\.,]+', texto_completo)
                    condominio = condominio.group(0).replace("Cond. R$ ", "").strip() if condominio else ''
                    iptu = iptu.group(0).replace("IPTU R$ ", "").strip() if iptu else ''
                except:
                    condominio = ''
                    iptu = ''

                # --- SALVAR RESULTADO ---
                dados_imoveis.append({
                    'Preço': preco,
                    'Rua': rua,
                    'Área': area,
                    'Quartos': quartos,
                    'Banheiros': banheiros,
                    'Vagas': vagas,
                    'Link': link,
                    'Bairro': bairro,
                    'Cidade': cidade,
                    'Condomínio': condominio,
                    'IPTU': iptu
                })

        # Avançar para próxima página
        try:
            botao_proxima = driver.find_element(By.CSS_SELECTOR, 'button[data-testid="next-page"]')
            botao_proxima.click()
            pagina_atual += 1
            time.sleep(tempo_espera_pagina)
        except Exception as e:
            print("Erro ao tentar avançar para a próxima página:", e)
            break

    return dados_imoveis

In [10]:
'''# Inicializa o navegador com o ChromeDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
# Acessa o site
driver.get(url)
time.sleep(5) # tempo de espera

dados = coletar_paginas_completas(driver)

df = pd.DataFrame(dados)
df.to_csv("imoveis_sjc.csv", index=False, encoding='utf-8-sig')

driver.quit()'''


Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 30 cards nesta página.

Página 4
Encontrados 30 cards nesta página.

Página 5
Encontrados 30 cards nesta página.

Página 6
Encontrados 30 cards nesta página.

Página 7
Encontrados 30 cards nesta página.

Página 8
Encontrados 30 cards nesta página.

Página 9
Encontrados 30 cards nesta página.

Página 10
Encontrados 30 cards nesta página.

Página 11
Encontrados 30 cards nesta página.

Página 12
Encontrados 30 cards nesta página.

Página 13
Encontrados 30 cards nesta página.

Página 14
Encontrados 30 cards nesta página.

Página 15
Encontrados 30 cards nesta página.

Página 16
Encontrados 30 cards nesta página.

Página 17
Encontrados 24 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" class="l-button l-button--appearance-iconButton l-button--conte

### Percorrendo URL's por bairros

#### URL's de São José dos Campos

In [7]:
eugenio_de_melo = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/eugenio-de-melo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Eug%C3%AAnio%20de%20Melo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EEugenio%20de%20Melo,-23.147078,-45.795956,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#jardim_ipe = "" ESTÁ COMO EUGENIO DE MELO
#nessa url ficou itapua e itapuâ juntos na pesquisa
jardim_itapua = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-itapoa/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Itapoa,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Itapoa,-23.140225,-45.779556,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Itapu%C3%A3,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Itapua,-23.140225,-45.779556,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_armando_moreira_righi = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-armando-moreira-righi/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Armando%20Moreira%20Righi,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Armando%20Moreira%20Righi,-23.136643,-45.765723,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conjunto_residencial_galo_branco = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/conjunto-residencial-galo-branco/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%20Galo%20Branco,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%20Galo%20Branco,-23.135348,-45.768685,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_sao_jose = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sao-jose/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20S%C3%A3o%20Jos%C3%A9,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Jose,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conj_hab_jardim_sao_jose_ll_leste = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sao-jose-leste/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20S%C3%A3o%20Jos%C3%A9%20-%20Leste,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Jose%20Leste,-23.21984,-45.891566,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Sao%20Jose%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Jose%20II,-23.170565,-45.779,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_americano = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-americano/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Americano,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Americano,-23.181223,-45.811959,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#jardim_coqueiro = "" bairro novo de 2023, não tem imóveis
jardim_motorama = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-motorama/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Motorama,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Motorama,-23.17439,-45.824414,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_nova_detroit = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-nova-detroit/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Nova%20Detroit,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Nova%20Detroit,-23.167026,-45.810179,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_nova_florida = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-nova-florida/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Nova%20Florida,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Nova%20Florida,-23.179226,-45.810476,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_pararangaba = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-pararangaba/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Pararangaba,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Pararangaba,-23.174106,-45.810179,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_rodolfo = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-rodolfo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Rodolfo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Rodolfo,-23.175739,-45.813145,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_ines_l = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-ines-i/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20In%C3%AAs%20I,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Ines%20I,-23.168338,-45.797135,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_ines_ll = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-ines-ii/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20In%C3%AAs%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Ines%20II,-23.170078,-45.805435,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_ines_lll = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-ines-iii/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20In%C3%AAs%20III,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Ines%20III,-23.16972,-45.78765,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_sao_vicente = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sao-vicente/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20S%C3%A3o%20Vicente,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Vicente,-23.169528,-45.817296,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_ana_maria = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-ana-maria/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Ana%20Maria,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Ana%20Maria,-23.181914,-45.807214,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_campo_belo = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/condominio-residencial-campo-belo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Condominio%20Residencial%20Campo%20Belo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ECondominio%20Residencial%20Campo%20Belo,-23.226785,-45.886147,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_frei_galvao = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-frei-galvao/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Frei%20Galvao,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Frei%20Galvao,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_castanheira = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-castanheira/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Castanheira,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Castanheira,-23.185457,-45.792587,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Castanheira%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Castanheira%20II,-23.172967,-45.78296,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_cerejeiras = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-cerejeiras/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Cerejeiras,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Cerejeiras,-23.194545,-45.78963,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_nova_michigan = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-nova-michigan/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Nova%20Michigan,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Nova%20Michigan,-23.187181,-45.795356,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Nova%20Michigan%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Nova%20Michigan%20II,-23.190573,-45.791549,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_paineiras_l_e_ll = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-das-paineiras-i/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20das%20Paineiras%20I,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20das%20Paineiras%20I,-23.208237,-45.786464,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20das%20Paineiras%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20das%20Paineiras%20II,-23.208237,-45.786464,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_san_rafael = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-san-rafael/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20San%20Rafael,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20San%20Rafael,-23.189886,-45.792985,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_nova_esperanca = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-nova-esperanca/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Nova%20Esperan%C3%A7a,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Nova%20Esperanca,-23.199414,-45.778167,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_novo_horizonte = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-novo-horizonte/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Novo%20Horizonte,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Novo%20Horizonte,-23.195604,-45.784094,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_dom_bosco = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-dom-bosco/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Dom%20Bosco,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Dom%20Bosco,-23.204426,-45.792392,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
campos_de_sao_jose = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/campos-de-sao-jose/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Campos%20de%20Sao%20Jose,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ECampos%20de%20Sao%20Jose,-23.216567,-45.809027,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_helena = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/loteamento-jardim-helena/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Loteamento%20Jardim%20Helena,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ELoteamento%20Jardim%20Helena,-23.21984,-45.891566,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Helena,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Helena,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_mariana_l_e_ll = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-mariana-ii/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Mariana%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Mariana%20II,-23.218878,-45.806621,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Mariana,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Mariana,-23.214774,-45.79832,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Mariana%20I,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Mariana%20I,-23.214774,-45.79832,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
pousada_do_vale = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/chacaras-pousada-do-vale/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Chacaras%20Pousada%20do%20Vale,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EChacaras%20Pousada%20do%20Vale,-23.216987,-45.791206,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_monterrey = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-monterrey/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Monterrey,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Monterrey,-23.225701,-45.794171,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
buquirinha = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/buquirinha/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Buquirinha,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EBuquirinha,-23.112577,-45.91586,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Buquirinha%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EBuquirinha%20II,-23.098159,-45.913759,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Mirante%20do%20Buquirinha,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EMirante%20do%20Buquirinha,-23.111657,-45.920015,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
alto_da_ponte = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/alto-da-ponte/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Alto%20da%20Ponte,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EAlto%20da%20Pte,-23.150061,-45.90132,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
altos_da_vila_paiva = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/altos-da-vila-paiva/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Altos%20da%20Vila%20Paiva,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EAltos%20da%20Vila%20Paiva,-23.142278,-45.914665,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
caete = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/recanto-caete/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Recanto%20Caet%C3%A9,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ERecanto%20Caete,-23.150516,-45.912883,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Altos%20do%20Caete%20I,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EAltos%20do%20Caete%20I,-23.141253,-45.924787,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Altos%20do%20Caete%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EAltos%20do%20Caete%20II,-23.141253,-45.924787,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conjunto_residencial_vila_leila = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/conjunto-residencial-vila-leila/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%20Vila%20Leila,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%20Vila%20Leila,-23.154499,-45.90486,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_altos_de_santana = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-altos-de-santana/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Altos%20de%20Santana,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Altos%20de%20Santana,-23.162713,-45.912288,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_boa_vista = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-boa-vista/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Boa%20Vista,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Boa%20Vista,-23.176794,-45.886741,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_guimaraes = "Jardim Guimaraes, São José dos Campos"
jardim_minas_gerais = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-minas-gerais/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Minas%20Gerais,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Minas%20Gerais,-23.154993,-45.920609,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_matilde = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-matilde/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20Matilde,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Matilde,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_telespark = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-telespark/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Telespark,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Telespark,-23.16143,-45.90694,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
recanto_caete = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/recanto-caete/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Recanto%20Caet%C3%A9,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ERecanto%20Caete,-23.150516,-45.912883,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_caminho_das_montanhas = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-caminho-das-montanhas/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Caminho%20das%20Montanhas,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Caminho%20das%20Montanhas,-23.133093,-45.907831,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_independencia = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-independencia/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Independencia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Independencia,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_mantiqueira = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/condominio-residencial-mantiqueira/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Condom%C3%ADnio%20Residencial%20Mantiqueira,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ECondominio%20Residencial%20Mantiqueira,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_candida = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-candida/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20C%C3%A2ndida,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Candida,-23.440312,-46.361269,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_dirce = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-dirce/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Dirce,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Dirce,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_leila = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/conjunto-residencial-vila-leila/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%20Vila%20Leila,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%20Vila%20Leila,-23.154499,-45.90486,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#vila_leonidia = "" não achei esse bairro no site, é bairro próx a vila paiva
#vila_mariteia = "" também não achei, nem no google maps
vila_monte_alegre = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-monte-alegre/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Monte%20Alegre,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Monte%20Alegre,-23.158235,-45.904563,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#vila_nossa_sra_das_gracas = "" achei no google, bairro pequeno e distante, mas não nos sites de imóveis
vila_paiva = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/altos-da-vila-paiva/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Altos%20da%20Vila%20Paiva,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EAltos%20da%20Vila%20Paiva,-23.142278,-45.914665,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Paiva,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Paiva,-23.139466,-45.911694,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#vila_santarem = "" também não achei, é zona norte novamente, apenas no google maps
vila_sao_geraldo = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-sao-geraldo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20S%C3%A3o%20Geraldo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Sao%20Geraldo,-23.147703,-45.909911,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_sinha = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-sinha/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Sinha,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Sinha,-23.157695,-45.907237,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Favela%20Vila%20Sinh%C3%A1,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EFavela%20Vila%20Sinha,-23.187528,-45.892087,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_unidos = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-unidos/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Unidos,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Unidos,-23.157064,-45.915557,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_venezziani = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-veneziani/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Veneziani,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Veneziani,-23.150061,-45.90132,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_por_do_sol = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-por-do-sol/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20P%C3%B4r%20do%20Sol,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Por%20do%20Sol,-23.246749,-45.939633,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
chacara_sao_jose = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/chacara-sao-jose/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Chacara%20Sao%20Jose,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EChacara%20Sao%20Jose,-23.207978,-45.851116,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_colorado = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-colorado/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Colorado,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Colorado,-23.213953,-45.854677,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_da_granja = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-da-granja/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20da%20Granja,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20da%20Granja,-23.204959,-45.857645,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_souto = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-souto/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Souto,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Souto,-23.200689,-45.860613,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_uira = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-uira/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Uira,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Uira,-23.210756,-45.852303,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_martim_cerere = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-martim-cerere/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Martim%20Cerere,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Martim%20Cerere,-23.211227,-45.856161,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_santa_rita = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-santa-rita/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Santa%20Rita,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Santa%20Rita,-23.200858,-45.849335,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#residencial_bell_park = "" não foi encontrado :(
residencial_cambui = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-cambui/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Cambui,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Cambui,-23.196991,-45.862691,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_flamboyant = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-residencial-flamboyant/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Residencial%20Flamboyant,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Residencial%20Flamboyant,-23.216624,-45.850522,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%20Flamboyant,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%20Flamboyant,-23.216624,-45.850522,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_sao_francisco = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-sao-francisco/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20S%C3%A3o%20Francisco,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Sao%20Francisco,-23.21016,-45.842213,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_sao_benedito = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-sao-benedito/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20S%C3%A3o%20Benedito,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Sao%20Benedito,-23.193986,-45.860019,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#conj_hab_policia_militar = "" não encontrado
#conj_res_nosso_teto = "" não encontrado
jardim_do_lago = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-do-lago/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20do%20Lago,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20do%20Lago,-23.241618,-45.841027,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_fe = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-fe/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20F%C3%A9,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Fe,-23.249538,-45.8434,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_julia = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-julia/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20J%C3%BAlia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Julia,-23.234833,-45.825951,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_luzia = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-luzia/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20Luzia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Luzia,-23.237918,-45.833811,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santa_rosa = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santa-rosa/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santa%20Rosa,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santa%20Rosa,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_santo_onofre = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-santo-onofre/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Santo%20Onofre,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Santo%20Onofre,-23.249698,-45.832126,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_sao_judas_tadeu = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sao-judas-tadeu/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20S%C3%A3o%20Judas%20Tadeu,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Judas%20Tadeu,-23.267757,-45.811366,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_sao_leopoldo = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sao-leopoldo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20S%C3%A3o%20Leopoldo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Leopoldo,-23.243649,-45.825007,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_jatoba = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-jatoba/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Jatob%C3%A1,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Jatoba,-23.25351,-45.826194,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_juritis = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-juritis/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Juritis,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Juritis,-23.255542,-45.828278,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_adriana = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-adriana/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Adriana,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Adriana,-23.256485,-45.817889,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_iracema = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-iracema/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Iracema,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Iracema,-23.241104,-45.81611,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_rica = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-rica/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20Rica,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20Rica,-23.245413,-45.834203,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
bosque_dos_ipes = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-bosque-dos-ipes/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Bosque%20dos%20Ip%C3%AAs,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Bosque%20dos%20Ipes,-23.256114,-45.895651,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
campo_dos_alemaes = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/campo-dos-alemaes/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Campo%20dos%20Alem%C3%A3es,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ECampo%20dos%20Alemaes,-23.269526,-45.896839,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
cidade_morumbi_e_conj_res = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/cidade-morumbi/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Cidade%20Morumbi,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ECidade%20Morumbi,-23.252295,-45.901592,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Morumbi,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Morumbi,-23.252295,-45.901592,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conj_hab_dom_pedro_l_e_ll = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/conjunto-residencial-dom-pedro-i/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%20Dom%20Pedro%20I,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%20Dom%20Pedro%20I,-23.279631,-45.890305,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Habitacional%20Dom%20Pedro%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Habitacional%20Dom%20Pedro%20II,-23.273234,-45.885553,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#conj_hab_elmano_f_veloso = "" Não encontrado, nem no google maps
conj_res_31_de_marco = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/conjunto-residencial-31-de-marco/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%2031%20de%20Mar%C3%A7o,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%2031%20de%20Marco,-23.246637,-45.914071,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Resid%2031%20de%20Marco,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Resid%2031%20de%20Marco,-23.246637,-45.914071,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conj_res_morada_do_sol = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/morada-do-sol/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Morada%20do%20Sol,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EMorada%20do%20Sol,-23.254715,-45.916057,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Loteamento%20Conjunto%20Morada%20do%20Sol%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ELoteamento%20Conjunto%20Morada%20do%20Sol%20II,-23.254715,-45.916057,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Residencial%20Morada%20do%20Sol,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Residencial%20Morada%20do%20Sol,-23.254562,-45.916448,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conj_res_primavera = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/primavera-1b/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Primavera%201B,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EPrimavera%201B,-23.21984,-45.891566,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Primavera%201A,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EPrimavera%201A,-23.21984,-45.891566,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Primavera%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Primavera%20II,-23.201406,-45.760392,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Primavera%20I,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EPrimavera%20I,-23.21984,-45.891566,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Primavera%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EPrimavera%20II,-23.213953,-45.854677,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Primavera%20II%20A,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Primavera%20II%20A,-23.201406,-45.760392,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conj_res_recanto_eucaliptos = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/recanto-dos-eucaliptos/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Recanto%20dos%20Eucaliptos,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ERecanto%20dos%20Eucaliptos,-23.224643,-45.850312,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conj_res_recanto_pinheiros = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/recanto-dos-pinheiros/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Recanto%20dos%20Pinheiros,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3ERecanto%20dos%20Pinheiros,-23.21984,-45.891566,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
conjunto_habitacional_papa_joao_paulo_ll = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/conjunto-papa-joao-paulo-ii/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Conjunto%20Papa%20Joao%20Paulo%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EConjunto%20Papa%20Joao%20Paulo%20II,-23.278377,-45.886741,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_colonial = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-colonial/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Col%C3%B4nial,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Colonial,-23.284429,-45.893869,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_cruzeiro_do_sul = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-cruzeiro-do-sul/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Cruzeiro%20do%20Sul,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Cruzeiro%20do%20Sul,-23.289044,-45.888523,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_dos_bandeirantes = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-dos-bandeirantes/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20dos%20Bandeirantes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20dos%20Bandeirantes,-23.285373,-45.901501,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_imperial = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-imperial/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Imperial,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Imperial,-23.285375,-45.901592,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_juliana = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-juliana/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Juliana,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Juliana,-23.261174,-45.902483,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_nova_republica = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-nova-republica/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Nova%20Rep%C3%BAblica,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Nova%20Republica,-23.290827,-45.898622,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_petropolis = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-petropolis/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Petr%C3%B3polis,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Petropolis,-23.241147,-45.91526,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_republica = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-republica/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Rep%C3%BAblica,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Republica,-23.291554,-45.895651,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Nova%20Rep%C3%BAblica,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Nova%20Republica,-23.290827,-45.898622,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
#jardim_santa_edwiges = "" não achou, mas é lá pra dps do CPO
jardim_sul = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sul/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Sul,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sul,-23.249753,-45.892681,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_terras_do_sul = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-terras-do-sul/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Terras%20do%20Sul,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Terras%20do%20Sul,-23.246627,-45.893869,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_vale_do_sol = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-vale-do-sol/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Vale%20do%20Sol,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Vale%20do%20Sol,-23.262797,-45.914665,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_veneza = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-veneza/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Veneza,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Veneza,-23.240659,-45.910505,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_dos_ypes = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-dos-ypes/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20dos%20Ypes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20dos%20Ypes,-23.263202,-45.895651,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_independencia = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-independencia/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Independ%C3%AAncia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Independencia,-23.252616,-45.917637,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_residencial_uniao = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-residencial-uniao/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Residencial%20Uni%C3%A3o,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Res%20Uniao,-23.262834,-45.911107,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_altos_do_bosque = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-altos-do-bosque/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Altos%20do%20Bosque,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Altos%20do%20Bosque,-23.266709,-45.893869,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_de_ville = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-de-ville/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20de%20Ville,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20de%20Ville,-23.251017,-45.916448,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
residencial_gazzo = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/residencial-gazzo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Residencial%20Gazzo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EResidencial%20Gazzo,-23.264073,-45.89981,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
vila_das_flores = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/vila-das-flores/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Vila%20das%20Flores,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EVila%20das%20Flores,-23.291407,-45.888523,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
jardim_mesquita = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-mesquita/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Mesquita,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Mesquita,-23.278169,-45.857645,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"
parque_interlagos = "https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/parque-interlagos/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Parque%20Interlagos,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EParque%20Interlagos,-23.276274,-45.842213,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o"


# Lista de URLs (uma por bairro)
lista_urls = [
    eugenio_de_melo,
    jardim_itapua,
    residencial_armando_moreira_righi,
    conjunto_residencial_galo_branco,
    jardim_sao_jose,
    conj_hab_jardim_sao_jose_ll_leste,
    jardim_americano,
    jardim_motorama,
    jardim_nova_detroit,
    jardim_nova_florida,
    jardim_pararangaba,
    jardim_rodolfo,
    jardim_santa_ines_l,
    jardim_santa_ines_ll,
    jardim_santa_ines_lll,
    jardim_sao_vicente,
    residencial_ana_maria,
    residencial_campo_belo,
    residencial_frei_galvao,
    jardim_castanheira,
    jardim_cerejeiras,
    jardim_nova_michigan,
    jardim_paineiras_l_e_ll,
    jardim_san_rafael,
    parque_nova_esperanca,
    parque_novo_horizonte,
    residencial_dom_bosco,
    campos_de_sao_jose,
    jardim_helena,
    jardim_mariana_l_e_ll,
    pousada_do_vale,
    vila_monterrey,
    buquirinha,
    alto_da_ponte,
    altos_da_vila_paiva,
    caete,
    conjunto_residencial_vila_leila,
    jardim_altos_de_santana,
    jardim_boa_vista,
    jardim_minas_gerais,
    jardim_santa_matilde,
    jardim_telespark,
    recanto_caete,
    residencial_caminho_das_montanhas,
    residencial_independencia,
    residencial_mantiqueira,
    vila_candida,
    vila_dirce,
    vila_leila,
    vila_monte_alegre,
    vila_paiva,
    vila_sao_geraldo,
    vila_sinha,
    vila_unidos,
    vila_venezziani,
    jardim_por_do_sol,
    chacara_sao_jose,
    jardim_colorado,
    jardim_da_granja,
    jardim_souto,
    jardim_uira,
    parque_martim_cerere,
    parque_santa_rita,
    residencial_cambui,
    residencial_flamboyant,
    residencial_sao_francisco,
    vila_sao_benedito,
    jardim_do_lago,
    jardim_santa_fe,
    jardim_santa_julia,
    jardim_santa_luzia,
    jardim_santa_rosa,
    jardim_santo_onofre,
    jardim_sao_judas_tadeu,
    jardim_sao_leopoldo,
    residencial_jatoba,
    residencial_juritis,
    vila_adriana,
    vila_iracema,
    vila_rica,
    bosque_dos_ipes,
    campo_dos_alemaes,
    cidade_morumbi_e_conj_res,
    conj_hab_dom_pedro_l_e_ll,
    conj_res_31_de_marco,
    conj_res_morada_do_sol,
    conj_res_primavera,
    conj_res_recanto_eucaliptos,
    conj_res_recanto_pinheiros,
    conjunto_habitacional_papa_joao_paulo_ll,
    jardim_colonial,
    jardim_cruzeiro_do_sul,
    jardim_dos_bandeirantes,
    jardim_imperial,
    jardim_juliana,
    jardim_nova_republica,
    jardim_petropolis,
    jardim_republica,
    jardim_sul,
    jardim_terras_do_sul,
    jardim_vale_do_sol,
    jardim_veneza,
    parque_dos_ypes,
    parque_independencia,
    parque_residencial_uniao,
    residencial_altos_do_bosque,
    residencial_de_ville,
    residencial_gazzo,
    vila_das_flores,
    jardim_mesquita,
    parque_interlagos
]


In [8]:
# Lista final com todos os dados
todos_os_dados = []

for url in lista_urls:
    print(f"\n Coletando dados da URL: {url}")

    # Inicia o driver novo
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # Faz um reset de estado do navegador
    driver.get("about:blank")
    time.sleep(1)

    # Carrega a URL do bairro
    driver.get(url)
    time.sleep(7)  # Espera extra para garantir o carregamento completo

    # Coleta os dados normalmente
    dados = coletar_paginas_completas(driver)
    todos_os_dados.extend(dados)

    # Encerra navegador
    driver.quit()

# Cria e salva o DataFrame
df = pd.DataFrame(todos_os_dados)
df.to_csv("imoveis_sjc.csv", index=False, encoding='utf-8-sig')
print("\n CSV salvo com sucesso!")


🌐 Coletando dados da URL: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/eugenio-de-melo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Eug%C3%AAnio%20de%20Melo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EEugenio%20de%20Melo,-23.147078,-45.795956,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o

Página 1
Encontrados 44 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 30 cards nesta página.

Página 4
Encontrados 30 cards nesta página.

Página 5
Encontrados 30 cards nesta página.

Página 6
Encontrados 30 cards nesta página.

Página 7
Encontrados 30 cards nesta página.

Página 8
Encontrados 30 cards nesta página.

Página 9
Encontrados 30 cards nesta página.

Página 10
Encontrados 30 cards nesta página.

Página 11
Encontrado


Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 26 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" class="l-button l-button--appearance-iconButton l-button--context-primary l-button--no-label l-button--size-regular" disabled="">...</button> is not clickable at point (1116, 800). Other element would receive the click: <nav data-testid="l-pagination" class="l-pagination l-pagination--numbered">...</nav>
  (Session info: chrome=136.0.7103.93)
Stacktrace:
	GetHandleVerifier [0x011DFC53+61635]
	GetHandleVerifier [0x011DFC94+61700]
	(No symbol) [0x010005D3]
	(No symbol) [0x0104ECB0]
	(No symbol) [0x0104D054]
	(No symbol) [0x0104ABF7]
	(No symbol) [0x01049EFB]
	(No symbol) [0x0103E5A5]
	(No symbol) [0x0106D29C]
	(No symbol) [0x0103E034]
	(No symbol) [0x0106D514]
	(No symbol) [0x0108E61B]
	(No symbol) [0x0106D096]
	(No symbol) [0x0103C840]
	


🌐 Coletando dados da URL: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-rodolfo/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Rodolfo,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Rodolfo,-23.175739,-45.813145,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o

Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 11 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" class="l-button l-button--appearance-iconButton l-button--context-primary l-button--no-label l-button--size-regular" disabled="">...</button> is not clickable at point (1142, 800). Othe


🌐 Coletando dados da URL: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-sao-vicente/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20S%C3%A3o%20Vicente,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Sao%20Vicente,-23.169528,-45.817296,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o

Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 30 cards nesta página.

Página 4
Encontrados 30 cards nesta página.

Página 5
Encontrados 3 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" class="l-button l-button--appearance-iconButton l-button--context-primary l


🌐 Coletando dados da URL: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-castanheira/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Castanheira,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Castanheira,-23.185457,-45.792587,;,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Castanheira%20II,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Castanheira%20II,-23.172967,-45.78296,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o

Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 8 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" clas

...


Página 1
Encontrados 14 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: no such element: Unable to locate element: {"method":"css selector","selector":"button[data-testid="next-page"]"}
  (Session info: chrome=136.0.7103.93); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x011DFC53+61635]
	GetHandleVerifier [0x011DFC94+61700]
	(No symbol) [0x010005D3]
	(No symbol) [0x0104899E]
	(No symbol) [0x01048D3B]
	(No symbol) [0x01090E12]
	(No symbol) [0x0106D2E4]
	(No symbol) [0x0108E61B]
	(No symbol) [0x0106D096]
	(No symbol) [0x0103C840]
	(No symbol) [0x0103D6A4]
	GetHandleVerifier [0x01464573+2701795]
	GetHandleVerifier [0x0145FCF6+2683238]
	GetHandleVerifier [0x0147AA3E+2793134]
	GetHandleVerifier [0x011F6915+155013]
	GetHandleVerifier [0x011FCFFD+181357]
	GetHandleVerifier [0x011E74A8+92440]
	GetHandleVerifier [0x011E7650+92864]
	GetH


🌐 Coletando dados da URL: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-cruzeiro-do-sul/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Cruzeiro%20do%20Sul,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Cruzeiro%20do%20Sul,-23.289044,-45.888523,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o

Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 30 cards nesta página.

Página 4
Encontrados 10 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" class="l-button l-button--appearance-iconButton l-button--context-primary l-button--no-label l-button--size-

Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página" type="button" class="l-button l-button--appearance-iconButton l-button--context-primary l-button--no-label l-button--size-regular" disabled="">...</button> is not clickable at point (1142, 800). Other element would receive the click: <nav data-testid="l-pagination" class="l-pagination l-pagination--numbered">...</nav>
  (Session info: chrome=136.0.7103.93)
Stacktrace:
	GetHandleVerifier [0x011DFC53+61635]
	GetHandleVerifier [0x011DFC94+61700]
	(No symbol) [0x010005D3]
	(No symbol) [0x0104ECB0]
	(No symbol) [0x0104D054]
	(No symbol) [0x0104ABF7]
	(No symbol) [0x01049EFB]
	(No symbol) [0x0103E5A5]
	(No symbol) [0x0106D29C]
	(No symbol) [0x0103E034]
	(No symbol) [0x0106D514]
	(No symbol) [0x0108E61B]
	(No symbol) [0x0106D096]
	(No symbol) [0x0103C840]
	(No symbol) [0x0103D6A4]
	GetHandleVerifier [0x01464573+2701795]
	GetHandleVerifier [0x014


🌐 Coletando dados da URL: https://www.vivareal.com.br/venda/sp/sao-jose-dos-campos/bairros/jardim-vale-do-sol/apartamento_residencial/?transacao=venda&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Jos%C3%A9%20dos%20Campos,Bairros,Jardim%20Vale%20do%20Sol,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Jose%20dos%20Campos%3EBarrios%3EJardim%20Vale%20do%20Sol,-23.262797,-45.914665,&tipos=apartamento_residencial,casa_residencial,condominio_residencial,cobertura_residencial,sobrado_residencial&pagina=1&precoMinimo=150000&ordem=Menor%20pre%C3%A7o

Página 1
Encontrados 30 cards nesta página.

Página 2
Encontrados 30 cards nesta página.

Página 3
Encontrados 30 cards nesta página.

Página 4
Encontrados 30 cards nesta página.

Página 5
Encontrados 30 cards nesta página.

Página 6
Encontrados 30 cards nesta página.

Página 7
Encontrados 26 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: element click intercepted: Element <button data-testid="next-page" aria-label="Próxima página"


Página 1
Encontrados 24 cards nesta página.
Erro ao tentar avançar para a próxima página: Message: no such element: Unable to locate element: {"method":"css selector","selector":"button[data-testid="next-page"]"}
  (Session info: chrome=136.0.7103.93); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x011DFC53+61635]
	GetHandleVerifier [0x011DFC94+61700]
	(No symbol) [0x010005D3]
	(No symbol) [0x0104899E]
	(No symbol) [0x01048D3B]
	(No symbol) [0x01090E12]
	(No symbol) [0x0106D2E4]
	(No symbol) [0x0108E61B]
	(No symbol) [0x0106D096]
	(No symbol) [0x0103C840]
	(No symbol) [0x0103D6A4]
	GetHandleVerifier [0x01464573+2701795]
	GetHandleVerifier [0x0145FCF6+2683238]
	GetHandleVerifier [0x0147AA3E+2793134]
	GetHandleVerifier [0x011F6915+155013]
	GetHandleVerifier [0x011FCFFD+181357]
	GetHandleVerifier [0x011E74A8+92440]
	GetHandleVerifier [0x011E7650+92864]
	GetH

In [18]:
driver.quit()

In [9]:
df

,Preço,Rua,Área,Quartos,Banheiros,Vagas,Link,Bairro,Cidade,Condomínio,IPTU
0,R$ 170.000,Rodovia Presidente Dutra,Tamanho do imóvel\n49 m²,Quantidade de quartos\n2,Quantidade de banheiros\n1,Quantidade de vagas de garagem\n1,https://www.vivareal.com.br/imovel/apartamento...,Apartamento para comprar em\nEugênio de Mello,São José dos Campos,,800
1,R$ 180.000,Rodovia Presidente Dutra,Tamanho do imóvel\n49 m²,Quantidade de quartos\n2,Quantidade de banheiros\n1,Quantidade de vagas de garagem\n1,None,Apartamento para comprar em\nEugênio de Mello,São José dos Campos,390,540
2,R$ 190.000,Rua Luiz Carlos Fraga e Silva,Tamanho do imóvel\n50 m²,Quantidade de quartos\n2,Quantidade de banheiros\n1,Quantidade de vagas de garagem\n1,https://www.vivareal.com.br/imovel/apartamento...,Apartamento para comprar em\nConjunto Residenc...,São José dos Campos,350,
3,R$ 202.000,Avenida das Oliveiras,Tamanho do imóvel\n42 m²,Quantidade de quartos\n2,Quantidade de banheiros\n1,Quantidade de vagas de garagem\n1,None,Apartamento para comprar em\nResidencial Frei ...,São José dos Campos,300,50
4,R$ 210.000,Avenida Dusmenil Santos Fernandes,Tamanho do imóvel\n47 m²,Quantidade de quartos\n2,Quantidade de banheiros\n1,Quantidade de vagas de garagem\n1,None,Apartamento para comprar em\nConjunto Residenc...,São José dos Campos,300,50
...,...,...,...,...,...,...,...,...,...,...,...
9024,R$ 3.500.000,Estrada Doutor Bezerra de Menezes,Tamanho do imóvel\n440 m²,Quantidade de quartos\n3,Quantidade de banheiros\n3,Quantidade de vagas de garagem\n18,https://www.vivareal.com.br/imovel/casa-de-con...,Casa de condomínio para comprar em\nJardim Tor...,São José dos Campos,910,
9025,R$ 5.800.000,Estrada Doutor Bezerra de Menezes,Tamanho do imóvel\n480 m²,Quantidade de quartos\n5,Quantidade de banheiros\n5-6,Quantidade de vagas de garagem\n12,None,Casa para comprar em\nJardim Torrão de Ouro,São José dos Campos,,1.800
9026,R$ 5.800.000,Estrada Doutor Bezerra de Menezes,Tamanho do imóvel\n480 m²,Quantidade de quartos\n5,Quantidade de banheiros\n7,Quantidade de vagas de garagem\n12,https://www.vivareal.com.br/imovel/casa-5-quar...,Casa para comprar em\nJardim Torrão de Ouro,São José dos Campos,,1.800
9027,R$ 8.000.000,Estrada Doutor Bezerra de Menezes,Tamanho do imóvel\n440 m²,Quantidade de quartos\n3,Quantidade de banheiros\n3,,https://www.vivareal.com.br/imovel/casa-de-con...,Casa de condomínio para comprar em\nJardim Tor...,São José dos Campos,900,
